# Pascal VOC 物体検出 — 学習・推論チュートリアル

PyTorch Lightning + Weights & Biases (wandb) を使った Faster R-CNN 物体検出モデルの  
**学習・推論の手順と証跡** をまとめた Notebook です。

**実行前提**
- Docker コンテナ起動済み（`/app` がプロジェクトルート）
- wandb 認証済み（未完了の場合: コンテナ内で `wandb login`）

---

## 手順

| ステップ | 内容 |
|---|---|
| Step 1 | 学習を実行する (`train.py`) |
| Step 2 | 最新チェックポイントを確認する |
| Step 3 | 推論を実行する (`inference.py`) |
| Step 4 | 推論結果画像を表示する |

## Step 1 — 学習の実行

`configs/train.yaml` の設定でそのまま学習します。  
`data.num_workers=0` は Notebook 環境での multiprocessing エラーを回避するためのオーバーライドです。

> 学習が完了すると `outputs/train/YYYY-MM-DD/HH-MM-SS/checkpoints/best-checkpoint.ckpt` が保存されます。

In [ ]:
import os
os.chdir("/app")  # Notebook からでも /app を起点にする

!PYTHONPATH=src python src/train.py data.num_workers=0

## Step 2 — 最新チェックポイントの確認

学習で保存された `.ckpt` ファイルを確認し、推論に使うパスを設定します。

In [ ]:
from pathlib import Path

# outputs/train/ 以下で最新のチェックポイントを自動取得
ckpts = sorted(Path("outputs/train").glob("**/best-checkpoint.ckpt"))
assert ckpts, "チェックポイントが見つかりません。Step 1 の学習を先に実行してください。"

CHECKPOINT = str(ckpts[-1])  # 最新を使用
print(f"使用するチェックポイント: {CHECKPOINT}")
print(f"ファイルサイズ          : {Path(CHECKPOINT).stat().st_size / 1e6:.1f} MB")

## Step 3 — 推論の実行

`inference.py` を使って VOC 2012 の画像を一括処理します。  
結果画像は `outputs/runs/YYYY-MM-DD/HH-MM-SS/` に自動保存されます。

In [ ]:
!PYTHONPATH=src python src/inference.py \
    --checkpoint {CHECKPOINT} \
    --image_dir data/VOCdevkit/VOC2012/JPEGImages \
    --num_images 10 \
    --score_thresh 0.5

## Step 4 — 推論結果の確認

最新の推論出力ディレクトリから結果画像を読み込んで表示します。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# 最新の推論出力ディレクトリを自動取得
pred_dirs = sorted(Path("outputs/runs").glob("*/*"))
assert pred_dirs, "推論結果が見つかりません。Step 3 を先に実行してください。"
latest_dir = pred_dirs[-1]

pred_images = sorted(latest_dir.glob("*_pred.jpg"))
print(f"結果ディレクトリ : {latest_dir}")
print(f"検出結果画像数   : {len(pred_images)} 枚")

# 最大 10 枚を 2 列で表示
n = min(len(pred_images), 10)
cols = 2
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(14, 7 * rows))
axes = axes.flatten()

for i, img_path in enumerate(pred_images[:n]):
    axes[i].imshow(mpimg.imread(img_path))
    axes[i].set_title(img_path.stem, fontsize=9)
    axes[i].axis("off")

# 余ったサブプロットを非表示
for j in range(n, len(axes)):
    axes[j].axis("off")

plt.suptitle("推論結果 — バウンディングボックス付き予測画像", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()